In [ ]:
import pandas as pd
import numpy as np

: 

In [ ]:
data = pd.read_csv('extract0.csv')
ruca = pd.read_excel('RUCA-codes-zipcode.xlsx', skiprows = 1, dtype={'ZIPCode': str})

In [ ]:
ruca.head()

In [ ]:
data.head()

**RUCA Codes**

<br> 1 = Metropolitan core: primary commuting flow is within an urban area of 50,000 or more people (metro UA) </br>
<br>2 = Metropolitan high commuting: primary commuting flow is 30 percent or more to a metro UA </br>
<br>3 = Metropolitan low commuting: primary commuting flow is 10 percent to 30 percent to a metro UA </br>
<br>4 = Micropolitan core: primary flow is within an urban area of 10,000 to 49,999 people (micro UA) </br>
<br>5 = Micropolitan high commuting: primary commuting flow is 30 percent or more to a micro UA </br>
<br>6 = Micropolitan low commuting: primary commuting flow is 10 percent to 30 percent to a micro UA </br>
<br>7 = Small town core: primary comuting flow is within an urban area of 9,999 or fewer people (small town UA) </br>
<br>8 = Small town high commuting: primary commmuting flow is 30 percent or more to a small town UA </br>
<br>9 = Small town low commuting: primary commuting flow is 10 percent to 30 percent to a small town UA </br>
<br>10 = Rural areas: primary commuting flow is to a tract outside an UA </br>


**I used the 4-level primary commuting category to classify rural/urban areas**

Metropolitan 1–3 

Micropolitan 4–6 

Small town 7–9 

Rural 10

**Codes 1 - 3 & 4 - 6 will be urban while codes 7 - 9 & 10 will be rural.**

# Mapping USPS Data Set to RUCA Data Set:

In [ ]:
ruca['ZIPCode'].unique()

In [ ]:
data['orgn_zip_5'] = data['orgn_zip_5'].astype(str)
data['destn_zip_5'] = data['destn_zip_5'].astype(str)
ruca['ZIPCode'] = ruca['ZIPCode'].astype(str)

merged = data.merge(ruca, left_on = 'orgn_zip_5', right_on = 'ZIPCode', how = 'left')
merged = merged.rename(columns={'PrimaryRUCA': 'Origin_RUCA'})

merged = merged.merge(ruca, left_on='destn_zip_5', right_on='ZIPCode',how='left', suffixes=('', '_dest'))
merged = merged.rename(columns={'PrimaryRUCA': 'Destination_RUCA'})

merged['Origin_Area_Type'] = np.select(
    [merged['Origin_RUCA'].isin([1,2,3,4,5,6]), merged['Origin_RUCA'].isin([7,8,9,10])],
    ['Urban', 'Rural'], default=None)

merged['Destination_Area_Type'] = np.select(
    [merged['Destination_RUCA'].isin([1,2,3,4,5,6]), merged['Destination_RUCA'].isin([7,8,9,10])],
    ['Urban', 'Rural'], default=None)

merged.head()

In [ ]:
merged[['orgn_zip_5', 'orgn_area_name', 'ZIPCode', 'Origin_RUCA', 'Origin_Area_Type', 'destn_zip_5', 'ZIPCode_dest','Destination_RUCA', 'Destination_RUCA','Destination_Area_Type']].head()

**Note:**

Some of the rows in orgn_zip_5 and destn_zip_5 have values of '-1' and 3 or 4 digit zipcodes instead of 5. The 3 or 4 digit zipcodes can be easily fixed using trailing zeros but we need to find a solution for the rows that have '-1'.